# Static correlations

Each eye-tracking metric is averaged across all stimulus scenes in a session, producing one value per session per metric. We then check whether these session-level averages correlate with depression severity measured by PHQ-9 and BDI.

Scenes are classified by valence pair type (`negative_vs_positive`, `negative_vs_neutral`, `neutral_vs_positive`), so per-valence metrics like `avg_dwell_neg__neg_vs_pos` mean "average negative dwell in scenes where the competitor was positive." Metrics that are independent of pairs (fixation counts, saccades, blinks, entropy) are averaged across all clean stimulus scenes regardless of pair type.

For each metric, three plots are shown: distribution across severity groups (violin), individual sessions (box + strip), and scatter with Spearman correlation.

**Structure:**
1. Build session-level dataset
2. PHQ-9 analysis: plots + correlation summary
3. BDI analysis: plots + correlation summary
4. Comparison: PHQ-9 vs BDI

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.features.session_aggregation import build_static_aggregation
from src.visualization.session import plot_severity_trend

## 1. Build session-level dataset

### 1.1 Filter scene-level data

Three filters are applied before aggregation:

- `scene_type == "stimulus"` - include only stimulus scenes and exclude fixation_cross scenes
- `scene_quality_valid == True` - standard eye-tracker quality gate applied by pipeline 04
- `scene_valence_pair` in `{negative_vs_positive, negative_vs_neutral, neutral_vs_positive}` - the three meaningful pair types
- `duration_ms` in [500, 10000] ms - exclude tracker outliers (scenes shorter than 0.5s or longer than 10s)

These threshold values are determined by the experimental setup.

In [0]:
VALID_PAIRS = ["negative_vs_positive", "negative_vs_neutral", "neutral_vs_positive"]
DURATION_MIN_MS = 500
DURATION_MAX_MS = 10000

scene_metrics = spark.table("anima.scene_metrics")

df_stimulus = (scene_metrics
    .filter(F.col("scene_type") == "stimulus")
    .filter(F.col("scene_quality_valid") == True)
    .filter(F.col("scene_valence_pair").isin(VALID_PAIRS))
    .filter(F.col("duration_ms").between(DURATION_MIN_MS, DURATION_MAX_MS)))

print(f"Clean stimulus scenes: {df_stimulus.count()}")

### 1.2 Build aggregated features

In [0]:
agg = build_static_aggregation()
print(f"Aggregation expressions: {len(agg.exprs)}")

### 1.3 Run aggregation

In [0]:
session_metrics = df_stimulus.groupBy("session_id").agg(*agg.exprs)
print(f"Sessions: {session_metrics.count()}")

## 2. Join with forms data

In [0]:
df_forms = spark.table("anima.forms").select(
    "session_id", "uid", "phq9_score", "phq9_severity", "bdi_score", "bdi_severity"
)
 
df_joined = session_metrics.join(df_forms, on="session_id", how="inner")
 
print(f"Sessions with both metrics and forms: {df_joined.count()}")
 
df = df_joined.toPandas()

## 3. Define severity group order

In [0]:
PHQ9_ORDER = ["minimal", "mild", "moderate", "moderately_severe", "severe"]
BDI_ORDER = ["minimal", "mild", "moderate", "severe"]
 
df["phq9_severity"] = pd.Categorical(df["phq9_severity"], categories=PHQ9_ORDER, ordered=True)
df["bdi_severity"] = pd.Categorical(df["bdi_severity"], categories=BDI_ORDER, ordered=True)
 
print("PHQ-9 severity distribution:")
print(df["phq9_severity"].value_counts().sort_index())
print()
print("BDI severity distribution:")
print(df["bdi_severity"].value_counts().sort_index())

## 4. Metrics to plot

In [0]:
metrics_to_plot = agg.all_columns
print(f"Total metrics to plot: {len(metrics_to_plot)}")

## 5. PHQ-9: metric plots

In [0]:
results_phq = []
for metric in metrics_to_plot:
    r = plot_severity_trend(
        df, metric,
        group_col="phq9_severity",
        score_col="phq9_score",
        score_label="PHQ-9 Score",
    )
    if r:
        results_phq.append(r)

## 6. PHQ-9: correlation summary

In [0]:
df_phq = pd.DataFrame(results_phq).sort_values("p_value")
df_phq["significant"] = df_phq["p_value"] < 0.05

print("PHQ-9 correlation summary (sorted by p-value):\n")
for _, row in df_phq.iterrows():
    star = " * " if row["significant"] else "   "
    print(f"{star} r={row['spearman_r']:+.3f}; p={row['p_value']:.2e}; {row['metric']}")

In [0]:
fig, ax = plt.subplots(figsize=(12, max(8, 0.25 * len(df_phq))))

colors = ["#d32f2f" if sig else "#90a4ae" for sig in df_phq["significant"]]

ax.barh(range(len(df_phq)), df_phq["spearman_r"], color=colors, edgecolor="white", height=0.7)

ax.set_yticks(range(len(df_phq)))
ax.set_yticklabels(df_phq["metric"].values, fontsize=9)

ax.axvline(x=0, color="black", linewidth=0.8)
ax.set_xlabel("Spearman r")
ax.set_title("PHQ-9: correlation with attention metrics (red = p < 0.05)")
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

## 7. BDI: metric plots

In [0]:
results_bdi = []
for metric in metrics_to_plot:
    r = plot_severity_trend(
        df, metric,
        group_col="bdi_severity",
        score_col="bdi_score",
        score_label="BDI Score",
    )
    if r:
        results_bdi.append(r)

## 8. BDI: correlation summary

In [0]:
df_bdi = pd.DataFrame(results_bdi).sort_values("p_value")
df_bdi["significant"] = df_bdi["p_value"] < 0.05

print("BDI correlation summary (sorted by p-value):")
print()
for _, row in df_bdi.iterrows():
    star = " * " if row["significant"] else "   "
    print(f"{star} r={row['spearman_r']:+.3f}; p={row['p_value']:.2e}; {row['metric']}")

In [0]:
fig, ax = plt.subplots(figsize=(12, max(8, 0.25 * len(df_bdi))))

colors = ["#d32f2f" if sig else "#90a4ae" for sig in df_bdi["significant"]]

ax.barh(range(len(df_bdi)), df_bdi["spearman_r"], color=colors, edgecolor="white", height=0.7)

ax.set_yticks(range(len(df_bdi)))
ax.set_yticklabels(df_bdi["metric"].values, fontsize=9)

ax.axvline(x=0, color="black", linewidth=0.8)
ax.set_xlabel("Spearman r")
ax.set_title("BDI: correlation with attention metrics (red = p < 0.05)")
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Comparison: PHQ-9 vs BDI

In [0]:
df_comparison = df_phq[["metric", "spearman_r", "p_value"]].merge(
    df_bdi[["metric", "spearman_r", "p_value"]],
    on="metric", suffixes=("_phq", "_bdi")
)

df_comparison["sig_phq"] = df_comparison["p_value_phq"] < 0.05
df_comparison["sig_bdi"] = df_comparison["p_value_bdi"] < 0.05

print(df_comparison[["metric", "spearman_r_phq", "p_value_phq", "spearman_r_bdi", "p_value_bdi"]]
      .sort_values("p_value_phq").to_string(index=False))

phq_only = (df_comparison["sig_phq"] & ~df_comparison["sig_bdi"]).sum()
bdi_only = (~df_comparison["sig_phq"] & df_comparison["sig_bdi"]).sum()
both = (df_comparison["sig_phq"] & df_comparison["sig_bdi"]).sum()

print()
print(f"Significant in PHQ-9 only: {phq_only}")
print(f"Significant in BDI only: {bdi_only}")
print(f"Significant in both: {both}")